# Notebook 07 - Inference Workflow and Production Usage

Final notebook for practical inference, output inspection, and deployment readiness.

## What Is This Technique? - Structured Legal Inference Pipeline

### Definition and Core Concepts
A production inference path that prioritizes fine-tuned adapters and falls back safely when adapter loading fails.

### Why Was This Technique Developed?
Production systems need resilient behavior, explicit output schema, and predictable latency.

### What Limitations of Traditional RAG Does It Solve?
RAG can feed better context but still needs a robust generator and output contract. This pipeline focuses on deterministic legal output structure.

### Architecture and Workflow Diagram Explanation
```mermaid
flowchart LR
A[Input legal text] --> B{Adapter available?}
B -->|Yes| C[Fine-tuned adapter inference]
B -->|No| D[Baseline generation fallback]
C --> E[Schema validation]
D --> E
E --> F[Executive summary + obligations + liabilities + risk]
```

### Component-by-Component Breakdown
1. Adapter-first inference selection.
2. Baseline fallback path.
3. Structured parser and schema validation.
4. Latency capture and persisted examples.

### When Should It Be Used in Real-World Systems?
Use in legal research assistants, contract review tools, compliance workflows, and regulatory intelligence systems.

### Advantages and Disadvantages
Advantages:
- Robust fallback behavior.
- Structured outputs ready for downstream systems.
- Supports local-only deployment.

Disadvantages:
- Small models can still miss subtle legal nuance.
- Latency/quality tradeoffs depend on model choice and hardware.

### Comparison Against Standard RAG and Other Implemented Variants
Compared with standard RAG-only responses, this output contract is directly consumable by policy engines, dashboards, and legal ops tooling.

### Implementation Details and Design Decisions Used in This Project
Inference now records latency and indicates strategy (`fine_tuned_adapter` or fallback).


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
ROOT = next((p for p in candidates if (p / 'scripts').exists() and (p / 'configs').exists()), cwd)
print('Project root:', ROOT)

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / script), *args]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))


In [ ]:
inference_path = ROOT / 'artifacts/inference/inference_examples.json'
if not inference_path.exists():
    run_py('scripts/run_inference.py', '--config', 'configs/default.yaml')
examples = json.loads(inference_path.read_text())
pd.DataFrame([
    {
        'example_id': x['example_id'],
        'strategy': x['strategy'],
        'model_used': x['model_used'],
        'risk_level': x['analysis']['risk_level'],
        'latency_seconds': x.get('latency_seconds', 0.0),
    }
    for x in examples
])

In [ ]:
examples[0]

In [ ]:
summary = {
    'strategy_counts': pd.Series([x['strategy'] for x in examples]).value_counts().to_dict(),
    'mean_latency_seconds': float(np.mean([x.get('latency_seconds', 0.0) for x in examples])),
    'max_latency_seconds': float(np.max([x.get('latency_seconds', 0.0) for x in examples])),
}
summary

## Post-Run Output Analysis

Interpret outputs with the following checklist:
1. Is the executive summary legally faithful and plain-English?
2. Are obligations/rights/liabilities supported by source wording?
3. Does risk classification include a clear rationale?
4. Are red flags actionable for legal review teams?

## Final Conclusion (Measured)

Use `summary`, `system_metrics.csv`, `scoreboard.csv`, `hallucination_report.json`, and `latency_report.json` to produce a run-specific conclusion. A strong conclusion should state:
- Which system won each key metric and by how much.
- Whether hallucination behavior improved or regressed.
- Whether current latency is acceptable for legal operations.
- Which concrete data/model changes are next for production hardening.